<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lab - Assessing and Fixing Agent Tools
#### Duration: 30 minutes
## Overview
In this lab, you will take a Unity Catalog function that works correctly but is poorly documented, see how it confuses an agent, and fix it. An agent decides when to call a tool, and what to pass it, based only on the tool's name, docstring, and parameter descriptions, so documentation is the interface between an agent and its tools. You will rewrite the function's documentation, re-register it to Unity Catalog, and verify the improvement through MCP and by testing it with a model.

Each step has a **TODO** with `<FILL_IN>` placeholders. The instructions above each TODO tell you what to fill in, and you can reveal the answer if you get stuck.

## Learning Objectives
By the end of this lab, you will be able to:
1. Identify documentation problems that keep an agent from using a UC function correctly
2. Rewrite a Python function with descriptive parameter names, type hints, and a structured docstring
3. Re-register an updated Python function to Unity Catalog with `DatabricksFunctionClient`
4. Discover and call a UC function through a Managed MCP server
5. Compare the arguments a model passes to a tool before and after improving the tool's documentation

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 6</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V6</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Environment Setup

Run the next cell to install the libraries for this lab and set up your catalog, schema, data, and the poorly documented `est_listing_value` function you will fix.

In [0]:
%run ./Includes/Classroom-Setup-3

Installing 8 package(s) with uv...
  pyyaml==6.0.3
  unitycatalog-ai[databricks]==0.3.2
  databricks-mcp==0.9.2
  databricks-sdk==0.145.0
  databricks-openai==0.17.1
  openai==3.22.1
  nest_asyncio==1.6.0
  mcp==1.30.0
Library installation complete.


Starting Demo Environment Setup
Using config: Includes/config/config-3.yaml

[0] Compute check passed: Serverless 6 detected.

[1] Setting up catalog...
Vocareum workspace detected.
  Catalog 'labuser17579966_1791895765' found. Using it.

[2] Setting up schema...
  Catalog : labuser17579966_1791895765
  Schema  : agent_training

[3] Setting up Unity Gateway model service...
  Model service created: labuser17579966_1791895765.agent_training.agent-training-model → databricks-gpt-5-2

[4] Processing datasets...
  Using Vocareum Delta Share: airbnb

[5] Registering 1 Python UC tool(s)...


/local_disk0/.ephemeral_nfs/envs/pythonEnv-9ae68b5c-c9fe-47c4-b4c8-e244856b8023/lib/python3.12/site-packages/databricks/connect/session.py:476: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


  Registered Python UC function: est_listing_value
Refreshing environment

Demo Environment Setup Complete


Catalog,"labuser17579966_1791895765<button onclick=""var t=document.createElement('textarea');t.value='labuser17579966_1791895765';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"
Schema,agent_trainingCopy
Table,"labuser17579966_1791895765.agent_training.sf_airbnb_listings<button onclick=""var t=document.createElement('textarea');t.value='labuser17579966_1791895765.agent_training.sf_airbnb_listings';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"
Python function,"labuser17579966_1791895765.agent_training.est_listing_value<button onclick=""var t=document.createElement('textarea');t.value='labuser17579966_1791895765.agent_training.est_listing_value';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"
Model service,"labuser17579966_1791895765.agent_training.agent-training-model<button onclick=""var t=document.createElement('textarea');t.value='labuser17579966_1791895765.agent_training.agent-training-model';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"


### A1. Lab Scenario

<div class="scn3"><style>.scn3{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif;color:#0b2026;width:100%;max-width:100%;box-sizing:border-box;margin:0;padding:8px}.scn3 .scen{background:#FFFFFF;color:#0b2026;padding:26px 30px;border-radius:12px;box-sizing:border-box;overflow:hidden;box-shadow:0 2px 16px rgba(27,49,57,0.10),0 1px 4px rgba(27,49,57,0.06);border:1px solid rgba(27,49,57,0.08)}.scn3 .kicker{font-size:14px;text-transform:uppercase;letter-spacing:1.2px;color:#5A6F77;margin-bottom:10px;font-weight:700;display:flex;align-items:center;gap:10px}.scn3 .body{font-size:15px;line-height:1.7;color:#0b2026}.scn3 .body p{margin:0 0 14px}.scn3 .body p:last-child{margin:0}.scn3 .tools{background:#F9F7F4;border:1px solid rgba(27,49,57,0.10);border-radius:8px;padding:12px 16px;margin:0 0 14px 0;font-family:ui-monospace,SFMono-Regular,Consolas,monospace;font-size:14px;line-height:1.9}.scn3 .fn{color:#0B8F5F}.scn3 .arg{color:#5A6F77}.scn3 .dsc{color:#5A6F77;opacity:.8}.scn3 code{background:rgba(27,49,57,0.07);padding:2px 6px;border-radius:4px;font-size:14px;font-family:ui-monospace,SFMono-Regular,Consolas,monospace;color:#0b2026}.scn3 .eq{background:rgba(0,169,114,.10);border-left:4px solid #00A972;padding:10px 14px;border-radius:0 6px 6px 0;margin:10px 0 14px;font-size:14px;font-style:italic;line-height:1.6;color:#0b2026}.scn3 .eq-q{background:#FFF6F4;border-left:4px solid #FF5F46;padding:10px 14px;border-radius:0 6px 6px 0;margin:10px 0 14px;font-size:14px;line-height:1.6;color:#0b2026}</style><div class="scen">
<div class="kicker"><svg width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="#5A6F77" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M12 8V4H8"/><rect x="4" y="8" width="16" height="12" rx="2"/><path d="M2 14h2"/><path d="M20 14h2"/><path d="M9 13v2"/><path d="M15 13v2"/></svg>Scenario</div>
<div class="body">
<p>Your team recently registered a Unity Catalog function called <code>est_listing_value</code> to help an agent estimate the annual revenue of an Airbnb listing. The logic is correct &#x2014; but when you give it to a model as a tool, the model consistently misuses it: passing values to the wrong parameters, inventing values the tool can't use, and returning a confident but wrong answer.</p>

<div class="tools"><div><span class="fn">est_listing_value</span><span class="arg">(p1, p2, p3)</span> <span class="dsc">&#x2014; "Calculates value."</span></div></div>

<p>The problem isn't the code &#x2014; it's the documentation. The function has meaningless parameter names (<code>p1</code>, <code>p2</code>, <code>p3</code>), a two-word description, and a docstring that is out of date: it says <code>p1</code> is the price, but the code uses <code>p1</code> as the number of bedrooms. An agent decides <strong>when</strong> to call a tool, <strong>what</strong> to pass it, and <strong>how</strong> to interpret the result based entirely on its name, docstring, and parameter descriptions. When those are unclear, the agent guesses. When they are wrong, the agent confidently follows them.</p>

<p>A user sends a perfectly reasonable request:</p>
<div class="eq">"I have a 2-bedroom entire-home Airbnb in SF priced at $250/night. What could I expect to make per year?"</div>

<p>But the agent sees only <code>p1: price</code>, <code>p2: count</code>, <code>p3: market</code> &#x2014; and produces something like:</p>
<div class="eq-q"><strong>Agent:</strong> Calls <code>est_listing_value(p1=250, p2=2, p3="San Francisco")</code> &#x2014; passing the price where the code expects the bedroom count, and a city where it expects a room type. Nothing fails: the tool reports about $9,798 a year instead of about $88,148.</div>

<p>You will diagnose the documentation problems, rewrite the function with descriptive names, type hints, and a structured docstring, re-register it to Unity Catalog, and verify the improvement through MCP and by testing it with a model.</p>
</div>
</div></div>

### A2. Inspect the Airbnb Dataset

As part of the classroom setup, the Airbnb dataset has been processed and stored as a Delta table within Unity Catalog. Run the next cell to preview the first few rows.

In [0]:
df = spark.read.table('sf_airbnb_listings')
display(df.limit(5))

id,name,neighbourhood,neighbourhood_cleansed,summary,price,room_type,listing_source_information
958,"Bright, Modern Garden Unit - 1BR/1B",Duboce Triangle,Western Addition,New update: the house next door is under construction and there is the possibility of noise from 7am - 5pm. Our rates are discounted during this time period. Our bright garden unit overlooks a grassy backyard area with fruit trees and native plants. It is an oasis in a big city. The apartment comfortably fits a couple or small family. It is located on a cul de sac street that ends at lovely Duboce Park.,170.0,Entire home/apt,"ID of the property: 958 Name of the property: Bright, Modern Garden Unit - 1BR/1B Summary of the property: New update: the house next door is under construction and there is the possibility of noise from 7am - 5pm. Our rates are discounted during this time period. Our bright garden unit overlooks a grassy backyard area with fruit trees and native plants. It is an oasis in a big city. The apartment comfortably fits a couple or small family. It is located on a cul de sac street that ends at lovely Duboce Park."
5858,Creative Sanctuary,Bernal Heights,Bernal Heights,null,235.0,Entire home/apt,null
7918,A Friendly Room - UCSF/USF - San Francisco,Cole Valley,Haight Ashbury,"Nice and good public transportation. 7 minutes walk to UCSF. 15 minutes walk to USF, ST Mary Hospital Room rental-sunny view room/sink/Wi Fi (inner sunset / UCSF) (map) The place is located within walking distance to Golden Gate Park, UCSF & USF",65.0,Private room,"ID of the property: 7918 Name of the property: A Friendly Room - UCSF/USF - San Francisco Summary of the property: Nice and good public transportation. 7 minutes walk to UCSF. 15 minutes walk to USF, ST Mary Hospital Room rental-sunny view room/sink/Wi Fi (inner sunset / UCSF) (map) The place is located within walking distance to Golden Gate Park, UCSF & USF"
8142,Friendly Room Apt. Style -UCSF/USF - San Francisco,Cole Valley,Haight Ashbury,"Nice and good public transportation. 7 minutes walk to UCSF. 15 minutes walk to USF, ST Mary Hospital Room rental-sunny view room/sink/Wi Fi (inner sunset / UCSF) (map) The place is located within walking distance to Golden Gate Park, UCSF & USF",65.0,Private room,"ID of the property: 8142 Name of the property: Friendly Room Apt. Style -UCSF/USF - San Francisco Summary of the property: Nice and good public transportation. 7 minutes walk to UCSF. 15 minutes walk to USF, ST Mary Hospital Room rental-sunny view room/sink/Wi Fi (inner sunset / UCSF) (map) The place is located within walking distance to Golden Gate Park, UCSF & USF"
8339,Historic Alamo Square Victorian,Western Addition/NOPA,Western Addition,"Pls email before booking. Interior featured in magazines. Listed historic with the City of SF. Steps away from the iconic Alamo Square ""Painted Ladies"". European feel, reminiscent of the Boutique Hotels in Paris, Berlin, Antwerp and London.",785.0,Entire home/apt,"ID of the property: 8339 Name of the property: Historic Alamo Square Victorian Summary of the property: Pls email before booking. Interior featured in magazines. Listed historic with the City of SF. Steps away from the iconic Alamo Square ""Painted Ladies"". European feel, reminiscent of the Boutique Hotels in Paris, Berlin, Antwerp and London."


### A3. Initialize the Databricks Function Client

**TODO:** Initialize the `DatabricksFunctionClient` for serverless compute. This client lets you register and execute UC functions programmatically.

In [0]:
## Import and initialize the DatabricksFunctionClient for serverless compute
# from unitycatalog.ai.core.databricks import <FILL_IN>

# client = <FILL_IN>
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient()

/local_disk0/.ephemeral_nfs/envs/pythonEnv-9ae68b5c-c9fe-47c4-b4c8-e244856b8023/lib/python3.12/site-packages/databricks/connect/session.py:476: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)



<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient()
```

</div>
</details>

## B. Examine a Poorly Documented Agent Tool

As part of the classroom setup, a Python function called `est_listing_value` has been registered to Unity Catalog. This function is intended to estimate the annual revenue of an Airbnb listing based on its characteristics, but its documentation is **vague and out of date**.

Let's look at the function as it currently exists.

### B1. Review the Bad Function

The function defined below has already been registered to Unity Catalog. Notice the problems: meaningless parameter names, a description that says almost nothing, and parameter descriptions that no longer match the code. Click each pill at the top to highlight a part of the code and read what's wrong with it.

<div id="py-code-explorer"></div>
<style>
  .sx-wrap-py {
    position: relative;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
    border: 1px solid #DCE0E2;
    border-radius: 12px;
    overflow: hidden;
    background: #F9F7F4;
    margin: 16px auto;
    max-width: 100%;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
  }
  .sx-wrap-py .sx-title {
    display: flex;
    justify-content: space-between;
    align-items: center;
    background: #EEEDE9;
    border-bottom: 1px solid #DCE0E2;
    padding: 10px 16px;
    font-size: 14px;
    font-weight: 700;
    color: #5A6F77;
    letter-spacing: 0.02em;
  }
  .sx-wrap-py .sx-rail {
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
    padding: 12px 16px;
    background: #F9F7F4;
    border-bottom: 1px solid #EEEDE9;
  }
  .sx-wrap-py .sx-chip {
    border: 1px solid #DCE0E2;
    background: #FFFFFF;
    border-radius: 999px;
    padding: 6px 12px;
    font-size: 14px;
    font-weight: 700;
    color: #0b2026;
    cursor: pointer;
    transition: background 0.12s, border-color 0.12s, color 0.12s;
  }
  .sx-wrap-py .sx-chip:hover,
  .sx-wrap-py .sx-chip.active {
    background: #E4F0FA;
    border-color: #2272B4;
    color: #1B5162;
  }
  .sx-wrap-py .sx-code {
    padding: 12px 0;
    overflow-x: auto;
    font-size: 14px;
    line-height: 1.6;
    background: #F9F7F4;
  }
  .sx-wrap-py .sx-code,
  .sx-wrap-py .sx-line-code {
    font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace !important;
  }
  .sx-wrap-py .sx-line {
    display: grid;
    grid-template-columns: 50px 1fr;
  }
  .sx-wrap-py .sx-line-no {
    color: #618794;
    text-align: right;
    padding: 0 12px;
    user-select: none;
    border-right: 1px solid #EEEDE9;
    font-weight: 700;
    font-size: 14px;
  }
  .sx-wrap-py .sx-line-code {
    padding: 0 16px;
    white-space: pre;
    color: #0b2026;
  }
  .sx-wrap-py .sx-hotspot { cursor: pointer; }
  .sx-wrap-py .sx-line.sx-active-line { background: #E4F0FA; }
  .sx-wrap-py .sx-active-line .sx-line-code {
    border-left: 3px solid #2272B4;
    padding-left: 13px;
  }
  .sx-wrap-py .sx-active-line .sx-line-no { color: #1B5162; }
  .sx-wrap-py .sx-overlay {
    display: none;
    position: absolute;
    right: 18px;
    top: 110px;
    width: 340px;
    max-width: calc(100% - 36px);
    background: #E4F0FA;
    border: 1px solid #4299E0;
    border-left: 5px solid #2272B4;
    border-radius: 10px;
    box-shadow: 0 6px 18px rgba(27,49,57,0.18), 0 16px 36px rgba(27,49,57,0.14);
    padding: 14px 16px;
    z-index: 5;
  }
  .sx-wrap-py .sx-overlay-title {
    font-weight: 800;
    margin-bottom: 8px;
    color: #1B5162;
    font-size: 16px;
  }
  .sx-wrap-py .sx-overlay-body {
    color: #0b2026;
    line-height: 1.5;
    font-size: 14px;
  }
  .sx-wrap-py .sx-overlay-body code {
    font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace;
    background: #FFFFFF;
    border: 1px solid #C4CCD6;
    color: #1B5162;
    padding: 1px 5px;
    border-radius: 4px;
    font-size: 14px;
  }
</style>
<script>
(function () {
  var config = {
    mountId: "py-code-explorer",
    title: "PYTHON",
    hotspots: [
      { title: "function name", startLine: 1, endLine: 1,
        description: "The tool name is the only helpful clue the agent gets. It hints at an estimate, but not of what, in what units, or for whom." },
      { title: "meaningless parameters", startLine: 2, endLine: 4,
        description: "<code>p1</code>, <code>p2</code>, and <code>p3</code> tell the agent nothing. The types help only a little: <code>p3</code> must be text, but the two floats are interchangeable." },
      { title: "return type", startLine: 5, endLine: 5,
        description: "<code>-> str</code> says the tool returns text, but not what the text contains." },
      { title: "two-word description", startLine: 6, endLine: 8,
        description: "<code>Calculates value.</code> is the tool description the model reads. It doesn't say what is calculated, or when to use the tool." },
      { title: "out-of-date Args", startLine: 9, endLine: 12,
        description: "These descriptions no longer match the code: <code>p1</code> is described as the price, but the code uses it as the bedroom count. <code>count</code> and <code>market</code> invite guesses, and a model will likely put a city in <code>p3</code>." },
      { title: "useless Returns", startLine: 13, endLine: 16,
        description: "<code>str: result</code> repeats the type and adds nothing about the value or its units." },
      { title: "where the docs bite", startLine: 18, endLine: 19,
        description: "The code multiplies <code>p2</code> as a nightly price and treats <code>p1</code> as a bedroom count, the reverse of what the docstring says." },
      { title: "silent fallback", startLine: 20, endLine: 25,
        description: "Any <code>p3</code> that isn't a known room type, such as <code>'San Francisco'</code>, silently falls through to the lowest multiplier instead of raising an error." },
      { title: "confident result", startLine: 26, endLine: 27,
        description: "The tool always returns a dollar amount, so a wrong call produces a believable wrong answer instead of an error." }
    ],
    source: "def est_listing_value(\n    p1: float,\n    p2: float,\n    p3: str\n) -> str:\n    \"\"\"\n    Calculates value.\n\n    Args:\n        p1: price\n        p2: count\n        p3: market\n\n    Returns:\n        str: result\n    \"\"\"\n\n    base_value = p2 * 365 * 0.7  # assume 70% occupancy\n    bedroom_factor = 1 + (p1 - 1) * 0.15\n    if p3.lower() == 'entire home/apt':\n        multiplier = 1.2\n    elif p3.lower() == 'private room':\n        multiplier = 0.8\n    else:\n        multiplier = 0.5\n    estimated = base_value * bedroom_factor * multiplier\n    return f\"Estimated annual revenue: ${estimated:,.2f}\""
  };
  function escapeHtml(v) {
    return String(v).replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/>/g,"&gt;");
  }
  var mount = document.getElementById(config.mountId);
  var lines = config.source.split("\n");
  var hotspots = config.hotspots;
  function hotspotFor(lineNo) {
    for (var i = 0; i < hotspots.length; i++) {
      if (lineNo >= hotspots[i].startLine && lineNo <= hotspots[i].endLine) return { h: hotspots[i], i: i };
    }
    return null;
  }
  var codeHtml = lines.map(function(line, idx) {
    var lineNo = idx + 1;
    var hs = hotspotFor(lineNo);
    var cls = hs ? "sx-line sx-hotspot sx-hotspot-" + hs.i : "sx-line";
    var attr = hs ? ' data-hotspot-index="' + hs.i + '"' : "";
    return '<div class="' + cls + '"' + attr + '><div class="sx-line-no">' + lineNo + '</div><div class="sx-line-code">' + escapeHtml(line) + '</div></div>';
  }).join("");
  var railHtml = '<div class="sx-rail">' + hotspots.map(function(h, i) {
    return '<button class="sx-chip" data-hotspot-index="' + i + '">' + escapeHtml(h.title) + '</button>';
  }).join("") + '</div>';
  mount.innerHTML = '<div class="sx-wrap-py"><div class="sx-title"><span>' + escapeHtml(config.title) + '</span></div>' + railHtml + '<div class="sx-code">' + codeHtml + '</div><div class="sx-overlay"><div class="sx-overlay-title"></div><div class="sx-overlay-body"></div></div></div>';
  var overlay = mount.querySelector(".sx-overlay");
  var overlayTitle = mount.querySelector(".sx-overlay-title");
  var overlayBody = mount.querySelector(".sx-overlay-body");
  var activeIdx = null;
  function clearActive() {
    mount.querySelectorAll(".sx-active-line").forEach(function(el) { el.classList.remove("sx-active-line"); });
    mount.querySelectorAll(".sx-chip.active").forEach(function(el) { el.classList.remove("active"); });
  }
  function activate(idx) {
    clearActive();
    if (idx === null) return;
    mount.querySelectorAll(".sx-hotspot-" + idx).forEach(function(el) { el.classList.add("sx-active-line"); });
    mount.querySelector('.sx-chip[data-hotspot-index="' + idx + '"]').classList.add("active");
  }
  function showHotspot(idx, triggerEl) {
    if (overlay.style.display === "block" && activeIdx === idx) {
      overlay.style.display = "none"; activeIdx = null; clearActive(); return;
    }
    var h = hotspots[idx];
    overlayTitle.textContent = h.title;
    overlayBody.innerHTML = h.description;
    overlay.style.display = "block";
    var wrap = mount.querySelector(".sx-wrap-py");
    var trigger = triggerEl || mount.querySelector(".sx-hotspot-" + idx);
    if (trigger) {
      var wrapRect = wrap.getBoundingClientRect();
      var trigRect = trigger.getBoundingClientRect();
      var top = trigRect.top - wrapRect.top;
      overlay.style.top = Math.max(18, Math.min(top, wrap.clientHeight - overlay.offsetHeight - 18)) + "px";
    }
    activeIdx = idx;
    activate(idx);
  }
  mount.querySelectorAll(".sx-hotspot").forEach(function(el) {
    el.addEventListener("click", function() { showHotspot(Number(el.dataset.hotspotIndex), el); });
  });
  mount.querySelectorAll(".sx-chip").forEach(function(chip) {
    chip.addEventListener("click", function() { showHotspot(Number(chip.dataset.hotspotIndex), null); });
  });
  mount.querySelector(".sx-wrap-py").addEventListener("click", function(e) {
    if (e.target.closest(".sx-hotspot") || e.target.closest(".sx-chip") || e.target.closest(".sx-overlay")) return;
    overlay.style.display = "none"; activeIdx = null; clearActive();
  });
  showHotspot(0, null);
})();
</script>

### B2. See How a Model Uses the Tool

Give the poorly documented tool to a model and see what it does with a realistic request. Run the next cell to connect to the model and define `ask_model`, the same helper as in the *Building Agent Tools on Databricks* demo, with one change: when you offer tools, it requires the model to call one, so you always see the arguments it chooses. It sends a question to the model, offers it the tools you name, and prints the tool calls the model wants to make without running them.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
We won't dive into this code much. The <em>Introduction to OpenAI Agents SDK on Databricks</em> demo, later in the course, covers connecting to a model and building agents in detail. For now, focus on the output.<br/><br/>The code calls a <strong>model service</strong>, <code>agent-training-model</code>, that the classroom setup created in your catalog and schema. A model service is our way of connecting to a model endpoint. We'll cover the details later in the course.
</p>
</div>

In [0]:
import json
import nest_asyncio
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient
from databricks_openai import DatabricksOpenAI

nest_asyncio.apply()
ws = WorkspaceClient()
test_mcp_client = DatabricksMCPClient(
    server_url=f"{ws.config.host}/api/2.0/mcp/functions/{catalog_name}/{schema_name}",
    workspace_client=ws,
)

# A model service is our way of connecting to a model endpoint (covered later in the course)
model_service = f"{catalog_name}.{schema_name}.agent-training-model"
llm_client = DatabricksOpenAI(workspace_client=ws, use_ai_gateway=True)

def ask_model(question, tool_names=None):
    """Send a question to the model, optionally offering UC functions as tools.

    Prints the model's answer, or the tool calls it wants to make and the
    arguments it would pass. The tools are not run.
    """
    offered = [t for t in test_mcp_client.list_tools() if t.name.split("__")[-1] in (tool_names or [])]
    tool_specs = [
        {"type": "function",
         "function": {"name": t.name, "description": t.description or "", "parameters": t.inputSchema}}
        for t in offered
    ]
    # Require a tool call so you always see the arguments the model picks
    extra = {"tools": tool_specs, "tool_choice": "required"} if tool_specs else {}
    response = llm_client.chat.completions.create(
        model=model_service,
        messages=[{"role": "user", "content": question}],
        **extra,
    )
    message = response.choices[0].message
    if message.tool_calls:
        print("The model wants to call:")
        for call in message.tool_calls:
            args = json.loads(call.function.arguments)
            print(f"  - {call.function.name.split('__')[-1]}({args})")
    else:
        print(message.content)

Now ask the question from the scenario. Look closely at the arguments. The model reads `p1: price`, `p2: count`, and `p3: market`, so it will likely pass the $250 price as `p1`, which the code treats as the bedroom count, and a city such as `"San Francisco"` as `p3`. Nothing would fail if the tool ran: a call like that returns about $9,798 a year, when the right answer is about $88,148. The model can't tell, because it trusts the documentation. The exact call can vary from run to run.

In [0]:
question = "I have a 2-bedroom entire-home Airbnb listing in San Francisco priced at $250. What could I expect to make from it per year?"

ask_model(question, tool_names=["est_listing_value"])

The model wants to call:
  - est_listing_value({'p1': 250, 'p2': 365, 'p3': 'San Francisco'})


<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Expected output</strong>
<pre style="margin: 10px 0; padding: 10px 12px; background: #ffffff; border: 1px solid #C6DCEA; border-radius: 6px; font-size: 12px; white-space: pre-wrap;">The model wants to call:
  - est_listing_value({'p1': 250, 'p2': 2, 'p3': 'San Francisco'})</pre>
<p style="margin: 8px 0 0 0; color: #333;">
The values are in the wrong parameters. The $250 price went into <code>p1</code>, which the code uses as the bedroom count, and a city went into <code>p3</code>, where the code expects a room type. If this call ran, the tool would return <strong>$9,798.42</strong>, a believable but wrong estimate. Your exact output can vary, for example <code>250.0</code> instead of <code>250</code>, or a different value for <code>p3</code>, but expect the price in <code>p1</code> and something other than a room type in <code>p3</code>.
</p>
</div>

## C. Fix the Function Description

The function logic is correct -- the problem is entirely in the documentation. An agent reads the function name, docstring, and parameter descriptions to decide:
1. **When** to call the function
2. **What** values to pass for each parameter
3. **How** to interpret the result

Without clear documentation, the agent is guessing.

### C1. Rewrite the Function with Proper Documentation

**TODO:** Redefine the function with:
- Descriptive, typed parameters: rename `p1`, `p2`, and `p3` to `num_bedrooms`, `nightly_price_usd`, and `room_type`, and give each a type hint. The function body already uses these names, and the test and MCP call below pass them by name.
- A clear docstring explaining the function's purpose and when to use it, and that matches what the code actually does
- Detailed parameter descriptions with types, units, and examples
- A clear return value description

In [0]:
## Redefine the function with an improved docstring
def est_listing_value(
    num_bedrooms: float,
    nightly_price_usd: float,
    room_type: str
) -> str:
    """
    Estimate the annual revenue for a San Francisco Airbnb listing based on
    its nightly price, number of bedrooms, and room type. Assumes 70%
    occupancy year-round.

    Use this tool when a user asks how much a listing could earn per year.

    Args:
        nightly_price_usd: The nightly price of the listing in US dollars (e.g. 250.0).
        num_bedrooms: The number of bedrooms in the listing (e.g. 1, 2, 3). A
            studio should be passed as 1. Each additional bedroom beyond
            the first increases the estimate by 15%.
        room_type: The Airbnb room type. Must be exactly one of:
            "Entire home/apt", "Private room", or "Shared room".

    Returns:
        A string describing the estimated annual revenue in USD.
    """
    base_value = nightly_price_usd * 365 * 0.7  # assume 70% occupancy
    bedroom_factor = 1 + (num_bedrooms - 1) * 0.15
    if room_type.lower() == 'entire home/apt':
        multiplier = 1.2
    elif room_type.lower() == 'private room':
        multiplier = 0.8
    else:
        multiplier = 0.5
    estimated = base_value * bedroom_factor * multiplier
    return f"Estimated annual revenue: ${estimated:,.2f}"


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
def est_listing_value(
    nightly_price_usd: float,
    num_bedrooms: float,
    room_type: str
) -> str:
    """
    Estimate the annual revenue for a San Francisco Airbnb listing based on
    its nightly price, number of bedrooms, and room type. Assumes 70%
    occupancy year-round.

    Use this tool when a user asks how much a listing could earn per year.

    Args:
        nightly_price_usd: The nightly price of the listing in US dollars (e.g. 250.0).
        num_bedrooms: The number of bedrooms in the listing (e.g. 1, 2, 3). A
            studio should be passed as 1. Each additional bedroom beyond
            the first increases the estimate by 15%.
        room_type: The Airbnb room type. Must be exactly one of:
            "Entire home/apt", "Private room", or "Shared room".

    Returns:
        A string describing the estimated annual revenue in USD.
    """
    base_value = nightly_price_usd * 365 * 0.7  # assume 70% occupancy
    bedroom_factor = 1 + (num_bedrooms - 1) * 0.15
    if room_type.lower() == 'entire home/apt':
        multiplier = 1.2
    elif room_type.lower() == 'private room':
        multiplier = 0.8
    else:
        multiplier = 0.5
    estimated = base_value * bedroom_factor * multiplier
    return f"Estimated annual revenue: ${estimated:,.2f}"
```

</div>
</details>

### C2. Test the Improved Function Locally

Before re-registering, verify the function works correctly at the notebook level.

In [0]:
result = est_listing_value(nightly_price_usd=250.0, num_bedrooms=2, room_type="entire home/apt")
print(result)

Estimated annual revenue: $88,147.50


### C3. Re-register the Function to Unity Catalog

**TODO:** Use `DatabricksFunctionClient` to re-register the improved function with `replace=True`.

In [0]:
function_info = client.create_python_function(
    func=est_listing_value,
    catalog=catalog_name,
    schema=schema_name,
    replace=True
)


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
function_info = client.create_python_function(
    func=est_listing_value,
    catalog=catalog_name,
    schema=schema_name,
    replace=True
)
```

</div>
</details>

## D. Verify the Improved Tool via MCP

In the demo, you saw how MCP allows agents to discover and call UC functions. Let's use MCP to verify that our re-registered function has the improved description and works correctly.

### D1. Discover the Tool

**TODO:** Initialize the MCP client and list available tools. Verify that `est_listing_value` appears with the improved description.

In [0]:
import nest_asyncio
nest_asyncio.apply()

# TODO
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient

ws = WorkspaceClient()
workspace_host = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}"

mcp_url = f"{workspace_host}/api/2.0/mcp/functions/{catalog_name}/{schema_name}"
mcp_client = DatabricksMCPClient(server_url=mcp_url, workspace_client=ws)

tools = mcp_client.list_tools()

for tool in tools:
    print(f"Tool: {tool.name}")
    print(f"  Description: {tool.description[:150]}")
    print()

Tool: labuser17579966_1791895765__agent_training__est_listing_value
  Description: Estimate the annual revenue for a San Francisco Airbnb listing based on its nightly price, number of bedrooms, and room type. Assumes 70% occupancy ye




<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
import nest_asyncio
nest_asyncio.apply()

from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient

ws = WorkspaceClient()
workspace_host = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}"

mcp_url = f"{workspace_host}/api/2.0/mcp/functions/{catalog_name}/{schema_name}"
mcp_client = DatabricksMCPClient(server_url=mcp_url, workspace_client=ws)

tools = mcp_client.list_tools()

for tool in tools:
    print(f"Tool: {tool.name}")
    print(f"  Description: {tool.description[:150]}")
    print()
```

</div>
</details>

### D2. Call the Tool via MCP

**TODO:** Execute `est_listing_value` through MCP to verify it works end-to-end.

In [0]:
result = mcp_client.call_tool(f"{catalog_name}__{schema_name}__est_listing_value", 
{
    "nightly_price_usd": 250.0, 
    "num_bedrooms":2, 
    "room_type":"entire home/apt"
    }
 )

print(f"MCP Result: {result}")

MCP Result: meta=None content=[TextContent(type='text', text='{"is_truncated":false,"columns":["output"],"rows":[["Estimated annual revenue: $88,147.50"]]}', annotations=None, meta=None)] structuredContent={'is_truncated': False, 'columns': ['output'], 'rows': [['Estimated annual revenue: $88,147.50']]} isError=False



<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
result = mcp_client.call_tool(f"{catalog_name}__{schema_name}__est_listing_value", 
{
    "nightly_price_usd": 250.0, 
    "num_bedrooms":2, 
    "room_type":"entire home/apt"
    }
 )

print(f"MCP Result: {result}")
```

</div>
</details>

## E. Validate the Improved Tool with a Model

Ask the model the same question as in B2. `ask_model` reads the tool's definition from the MCP server each time, so it now sees your improved names, type hints, and docstring.

In [0]:
ask_model(question, tool_names=["est_listing_value"])

The model wants to call:
  - est_listing_value({'num_bedrooms': 2, 'nightly_price_usd': 250, 'room_type': 'Entire home/apt'})


<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Expected output</strong>
<pre style="margin: 10px 0; padding: 10px 12px; background: #ffffff; border: 1px solid #C6DCEA; border-radius: 6px; font-size: 12px; white-space: pre-wrap;">The model wants to call:
  - est_listing_value({'nightly_price_usd': 250, 'num_bedrooms': 2, 'room_type': 'Entire home/apt'})</pre>
<p style="margin: 8px 0 0 0; color: #333;">
Each value is now in the right parameter. If your docstring lists the allowed room types, as the answer does, <code>room_type</code> is one of them. If this call ran, the tool would return <strong>$88,147.50</strong>, the correct estimate. Numbers may appear as <code>250.0</code> and <code>2.0</code>.
</p>
</div>

Compare the arguments with B2. The model now passes `nightly_price_usd`, `num_bedrooms`, and `room_type`, with values taken from the question, and uses a `room_type` from the list your docstring allows. The improved documentation gives the model the context it needs to use the tool correctly.

## Conclusion

In this lab you learned that **documentation is the interface** between an agent and its tools. The same function logic can be useless or powerful depending on how well it's documented. You also verified your tools work through MCP -- the same protocol that agents use in production to discover and call tools.

Here's what changed between B2 and E for the same question:

| | Bad tool (B2) | Fixed tool (E) |
|---|---|---|
| **Call the model makes** | `est_listing_value(p1=250, p2=2, p3="San Francisco")` | `est_listing_value(nightly_price_usd=250, num_bedrooms=2, room_type="Entire home/apt")` |
| **What the tool would return** | $9,798.42 (wrong) | $88,147.50 (correct) |
| **Why** | The out-of-date docstring sent each value to the wrong parameter | The names, types, and docstring match what the code does |

Key takeaways:
- Poor parameter names and docstrings make tools unusable by agents
- Out-of-date documentation is worse than none: the agent follows it confidently and gets a believable wrong answer
- Good documentation includes: clear names, detailed descriptions with examples, and business context
- MCP provides a standard way for agents to discover and call UC functions
- Always test tools via MCP, and with a model, before integrating them with agent code

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>